# Attention in Isolation

Picture three people, Ana, Ben and Cai, speaking in that order at a meeting. Each person writes a one-line summary of the meeting as they understand it when it’s their turn. The rules are:

- You may use what was said before you, and what you said yourself.
- You may not use anything said after you (you haven’t heard it yet).
- Your summary is the plain average of the points you’re allowed to use.

So Ana’s summary is just her own point, Ben’s averages Ana and Ben, and Cai’s averages all three.

## Init

In [25]:
from pathlib import Path
import torch
from torch import nn
import torch.functional as F
import numpy as np
import math

## Data

In [7]:
# X (3, 2), T = 3 tokens, d = 2
X = torch.tensor([
    [2.0, 0.0],     # x0
    [0.0, 4.0],     # x1
    [4.0, 2.0]      # x2
])

Averaging by hand:

In [3]:
out0 = torch.tensor([2, 0])
out1 = (out0 + torch.tensor([0, 4])) / 2
out2 = (out0 + torch.tensor([0, 4]) + torch.tensor([4, 2])) / 3

print("Out0:", out0)
print("Out1:", out1)
print("Out2:", out2)

Out0: tensor([2, 0])
Out1: tensor([1., 2.])
Out2: tensor([2., 2.])


Now the trick is to get all rows at once with one matrix multiply. Build a lower-triangular matrix of ones and normalise each row to sum to 1:

In [10]:
# tril := lower triangular
# triu := upper triangular
lower_tri = torch.tril(torch.ones(3, 3, dtype=torch.float16))
W = lower_tri / torch.sum(lower_tri, dim=1, keepdim=True)

out = W @ X.to(W.dtype)
print(f"W ({W.shape}) @ X ({X.shape}) -> out ({out.shape})")
print(out)

W (torch.Size([3, 3])) @ X (torch.Size([3, 2])) -> out (torch.Size([3, 2]))
tensor([[2., 0.],
        [1., 2.],
        [2., 2.]], dtype=torch.float16)


## Learned, data-dependent weights (Q, K, V)

Now give each speaker three things:

- **A question card (query)**: “What kind of earlier point would help me right now?”
- **A name badge (key)**: “This is what kind of point I am”, visible to everyone.
- **A folder of notes (value)**: “This is what I will hand over if you pick me.”

When it’s your turn, you compare your question card against the badge of every person you’re allowed to hear (yourself included). **The better the match, the more attention you give them**. You then blend their folders, not their badges, in proportion to that attention.

The key idea is to separate “how I’m found” (key) from “what I give” (value). A word might be easy to find because it’s a noun, yet contribute something quite different once found.

| In the analogy | In the model | Shape (T=3, d=2) |
|----------------|--------------|------------------|
| Everyone’s raw point | X | (3×2) |
| Turning a point into a question card | Q = X·W_Q | W_Q is (d×d_k) = (2×2), Q is (3×2) |
| Turning a point into a badge | K = X·W_K | W_K (2×2), K (3×2) |
| Turning a point into a folder | V = X·W_V | W_V (d×d_v) = (2×2), V (3×2) |
| Match between my question and your badge | Dot product q_i · k_j | S = Q·Kᵀ is (3×3) |
| Turning matches into attention shares | Row-wise softmax | W (3×3), each row sums to 1 |
| Blending the folders | W·V | out (3×2) |

In [11]:
X = torch.tensor([
    [1, 0],
    [0, 1],
    [1, 1]
])

W_Q = torch.tensor([[1, 0], [0, 1]])
W_K = torch.tensor([[0, 1], [1, 0]])
W_V = torch.tensor([[1, 2], [3, 0]])

In [13]:
Q = X @ W_Q
K = X @ W_K
V = X @ W_V

print("Q:", Q.shape)
print(Q)
print("\nK:", K.shape)
print(K)
print("\nV:", V.shape)
print(V)

Q: torch.Size([3, 2])
tensor([[1, 0],
        [0, 1],
        [1, 1]])

K: torch.Size([3, 2])
tensor([[0, 1],
        [1, 0],
        [1, 1]])

V: torch.Size([3, 2])
tensor([[1, 2],
        [3, 0],
        [4, 2]])


Calculate scores of Q (3 x 2) and K (3 x 2).

In [14]:
S = Q @ K.T

print("S:", S.shape)
print(S)

S: torch.Size([3, 3])
tensor([[0, 1, 1],
        [1, 0, 1],
        [1, 1, 2]])


Keep the lower triangle, then row-wise softmax

In [19]:
S = S.float()
mask = (lower_tri == 0)
masked_S = S.masked_fill(mask, float('-inf'))
softmax_S = torch.softmax(masked_S, dim=-1)

print("Low S:", masked_S.shape)
print(masked_S)
print("\nSoftmax S:", softmax_S.shape)
print(softmax_S)

Low S: torch.Size([3, 3])
tensor([[0., -inf, -inf],
        [1., 0., -inf],
        [1., 1., 2.]])

Softmax S: torch.Size([3, 3])
tensor([[1.0000, 0.0000, 0.0000],
        [0.7311, 0.2689, 0.0000],
        [0.2119, 0.2119, 0.5761]])


Blend the values. out = WV, (3x3)@(3x2) -> (3x2)

In [22]:
out = softmax_S @ V.to(softmax_S.dtype)

print("Out:", out.shape)
print(out)

Out: torch.Size([3, 2])
tensor([[1.0000, 2.0000],
        [1.5379, 1.4621],
        [3.1522, 1.5761]])


## Divide by $\sqrt{d_k}$

Imagine each score is the verdict of a panel of $d_k$ judges. Each judge gives a small random mark, positive or negative, and the verdict is the sum of all the marks. A panel of 4 judges rarely produces an extreme total. A panel of 400 often does, because random marks drift further from zero as you add more of them. When totals get extreme, the “attention shares” stop being shares and become a winner-takes-all vote.

In [24]:
import math
d_k = 2

S_scaled = S / math.sqrt(d_k)

print(S_scaled)

tensor([[0.0000, 0.7071, 0.7071],
        [0.7071, 0.0000, 0.7071],
        [0.7071, 0.7071, 1.4142]])


Then compute the mask, softmax as usual.

## Forward with NumPy

The pipeline

| Step | Operation | Shape |
|------|-----------|-------|
| 1 | Q = X W_Q | (T,d) @ (d, d_k) -> (T, d_k)|
| 2 | K = X W_K | (T, d) @ (d, d_k) -> (T, d_k)|
| 3 | V = X W_V | (T, d_v) |
| 4 | S = Q K^T / sqrt(d_k) | (T,d_k) @ (d_k,T) → (T,T) |
| 5 | mask: S[i,j] = −∞ where j > i | (T,T) |
| 6 | P = softmax over each row, after subtracting the row max | (T,T) |
| 7 | O = P V | (T,T) @ (T,d_v) → (T,d_v) |

In [28]:
def attention_forward(X, Wq, Wk, Wv, scale=None):
    Q = X @ Wq
    K = X @ Wk
    V = X @ Wv
    S = Q @ K.T
    T = X.shape[0]
    d_k = Wk.shape[1]

    if scale is None:
        scale = 1 / np.sqrt(d_k)

    S = S * scale

    mask = np.triu(np.ones((T, T), dtype=bool), k=1)
    S = np.where(mask, -np.inf, S)

    # Stability
    S -= S.max(axis=1, keepdims=True)

    # Softmax
    E = np.exp(S)
    P = E / E.sum(axis=1, keepdims=True)

    O = P @ V

    return O, (X, Q, K, V, P, Wq, Wk, Wv, scale)

In [ ]:
# def attention_forward(X, Wq, Wk, Wv, scale=None):
#     Q = X @ Wq
#     K = X @ Wk
#     V = X @ Wv
#     S = Q @ K.T
#     T = X.shape[0]
#     d_k = Wk.shape[1]

#     if scale is None:
#         scale = 1 / np.sqrt(d_k)

#     S = S * scale

#     mask = torch.triu(torch.ones((T,T), dtype=torch.bool), diagonal=1)
#     S = torch.where(mask, float('-inf'), S)

#     # Stability
#     S -= S.max(dim=1, keepdim=True).values

#     # Softmax
#     P = torch.softmax(torch.tensor(S), dim=1)


#     return X, Q, K, V, P, Wq, Wk, Wv, scale

## Backward Pass

The forward pass was a meeting that produced summaries. The backward pass starts from “the summary row t was wrong in this direction” and traces **blame backwards through the same steps**. Each step passes blame to its inputs in proportion to how much each input contributed. A step that mixes things (a matrix multiply) splits the blame by its weights. A step that competes (softmax) redistributes it among rivals.

For matrix product $C = AB$, given dC:
- $dA = dC \cdot B^T$
- $dB = A^T \cdot dC$

Start with $O = PV$, given dO.

In [ ]:
# def attention_backward(dO, cache):
#     X, Q, K, V, P, Wq, Wk, Wv, scale = cache
#     T = X.shape[0]
#     d_k = Wk.shape[1]

#     dP = dO @ V.T   # (T, T)
#     dV = P.T @ dO   # (T, d_v)

#     # P = softmax(S_stable)
#     # dS_stable = P * (dP - (dP * P^T)1)
#     # the stabilisation step (S -= S.max) has a derivative of 1
#     dS = P * (dP - torch.sum(dP * P, dim=1, keepdim=True))

#     # The mask blocked gradients from future elements (where values were -inf).
#     # We enforce the same mask on the gradients to ensure future positions stay 0.
#     mask = torch.triu(torch.ones((T, T), dtype=torch.bool), diagonal=1)
#     dS = dS.masked_fill(mask, 0.0)

#     # Scale backward
#     dS = dS * scale

#     # S = Q @ K.T
#     dQ = dS @ K
#     dK = dS.T @ Q

#     # Q = X @ Wq, K = X @ Wk, V = X @ Wv
#     dWq = X.T @ dQ
#     dWk = X.T @ dK
#     dWv = X.T @ dV

#     dX = (dQ @ Wq.T) + (dK @ Wk.T) + (dV @ Wv.T)
#     return dX, dWq, dWk, dWv

In [32]:
def attention_backward_numpy(dO, cache):
    X, Q, K, V, P, Wq, Wk, Wv, scale = cache
    T = X.shape[0]
    d_k = Wk.shape[1]

    # 1. Backward through: O = P @ V
    dP = dO @ V.T   # (T, T)
    dV = P.T @ dO   # (T, d_v)

    # 2. Backward through: P = softmax(S_stable)
    # Using np.sum with axis=1 and keepdims=True to handle broadcasting
    dS = P * (dP - np.sum(dP * P, axis=1, keepdims=True))

    # 3. Backward through Causal Mask & Scale
    # np.triu with k=1 isolates the future elements
    mask = np.triu(np.ones((T, T), dtype=bool), k=1)
    dS[mask] = 0.0

    # Scale backward
    dS = dS * scale

    # 4. Backward through: S = Q @ K.T
    dQ = dS @ K
    dK = dS.T @ Q

    # 5. Backward through the weight matrices
    dWq = X.T @ dQ
    dWk = X.T @ dK
    dWv = X.T @ dV

    # 6. Accumulate input gradient
    dX = (dQ @ Wq.T) + (dK @ Wk.T) + (dV @ Wv.T)
    
    return dX, dWq, dWk, dWv

## Test

In [31]:
scale=1.0
X = np.array([[1,0],[0,1],[1,1]])
Wq = np.eye(2)
Wk = np.array([[0,1],[1,0]])
Wv = np.array([[1,2],[3,0]])

O, cache = attention_forward(X, Wq, Wk, Wv, scale=scale)

print("O:", O)
print("(X, Q, K, V, P, Wq, Wk, Wv, scale):", cache)

O: [[1.         2.        ]
 [1.53788284 1.46211716]
 [3.15223377 1.57611688]]
(X, Q, K, V, P, Wq, Wk, Wv, scale): (array([[1, 0],
       [0, 1],
       [1, 1]]), array([[1., 0.],
       [0., 1.],
       [1., 1.]]), array([[0, 1],
       [1, 0],
       [1, 1]]), array([[1, 2],
       [3, 0],
       [4, 2]]), array([[1.        , 0.        , 0.        ],
       [0.73105858, 0.26894142, 0.        ],
       [0.21194156, 0.21194156, 0.57611688]]), array([[1., 0.],
       [0., 1.]]), array([[0, 1],
       [1, 0]]), array([[1, 2],
       [3, 0]]), 1.0)
